# Multi-Game Stint Validation and Lineup Integrity

This notebook validates the multi-game stint dataset created in Notebook 03.

The goal is to confirm that the reconstructed lineup data is structurally sound before it is used for lineup metrics, player impact, modeling, or dashboard development.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)

## Load multi-game reconstruction outputs

We load the multi-game stint dataset, substitution anomaly log, and game-level processing logs generated in Notebook 03.

In [2]:
stints_df = pd.read_pickle("../data/interim/stints_multigame.pkl")
sub_anomalies_df = pd.read_pickle("../data/interim/sub_anomalies_multigame.pkl")
game_logs_df = pd.read_csv("../data/interim/game_processing_logs.csv")

## Multi-game dataset overview

We first validate the overall size of the reconstructed dataset by checking the number of games and total stints.

In [3]:
print("Games processed:", stints_df["game_id"].nunique())
print("Total stints:", len(stints_df))

stints_df.head()

Games processed: 200
Total stints: 6293


,game_id,home_team_id,away_team_id,period,start_eventnum,start_clock,home_lineup,away_lineup,start_home_score,start_away_score,end_eventnum,end_clock,end_home_score,end_away_score,home_points,away_points,home_n_players,away_n_players,home_lineup_names,away_lineup_names
0,0020000001,1610612752,1610612755,1,1,12:00,"[164, 275, 369, 703, 779]","[238, 243, 248, 689, 727]",0,0,67,4:16,15,13,15,13,5,5,"[Chris Childs, Allan Houston, Charlie Ward, Kurt Thomas, Glen Rice]","[Tyrone Hill, Aaron McKie, George Lynch, Theo Ratliff, Eric Snow]"
1,0020000001,1610612752,1610612755,1,68,4:16,"[164, 275, 369, 703, 779]","[238, 243, 248, 689, 727]",15,14,69,4:16,15,14,0,0,5,5,"[Chris Childs, Allan Houston, Charlie Ward, Kurt Thomas, Glen Rice]","[Tyrone Hill, Aaron McKie, George Lynch, Theo Ratliff, Eric Snow]"
2,0020000001,1610612752,1610612755,1,70,4:16,"[164, 275, 369, 703, 779]","[238, 243, 248, 689, 727]",15,15,74,3:52,15,17,0,2,5,5,"[Chris Childs, Allan Houston, Charlie Ward, Kurt Thomas, Glen Rice]","[Tyrone Hill, Aaron McKie, George Lynch, Theo Ratliff, Eric Snow]"
3,0020000001,1610612752,1610612755,1,75,3:52,"[164, 275, 369, 703, 779]","[238, 243, 248, 689, 727]",15,18,80,3:16,15,20,0,2,5,5,"[Chris Childs, Allan Houston, Charlie Ward, Kurt Thomas, Glen Rice]","[Tyrone Hill, Aaron McKie, George Lynch, Theo Ratliff, Eric Snow]"
4,0020000001,1610612752,1610612755,1,82,3:10,"[164, 275, 369, 703, 779]","[238, 243, 248, 689, 727]",15,20,90,1:51,19,24,4,4,5,5,"[Chris Childs, Allan Houston, Charlie Ward, Kurt Thomas, Glen Rice]","[Tyrone Hill, Aaron McKie, George Lynch, Theo Ratliff, Eric Snow]"


## Global lineup integrity check

Each stint should contain exactly five players for the home team and five players for the away team.

In [4]:
stints_df["home_n_players"] = stints_df["home_lineup"].apply(len)
stints_df["away_n_players"] = stints_df["away_lineup"].apply(len)

stints_df[["home_n_players", "away_n_players"]].value_counts()

home_n_players  away_n_players
5               5                 6293
dtype: int64

## Game-level lineup integrity check

After checking lineup size globally, we verify that every individual game maintains five-player lineup integrity across all stints.

In [5]:
game_lineup_integrity = (
    stints_df
    .groupby("game_id")[["home_n_players", "away_n_players"]]
    .apply(lambda x: ((x == 5).all().all()))
    .reset_index(name="valid_lineups")
)

game_lineup_integrity["valid_lineups"].value_counts()

True    200
Name: valid_lineups, dtype: int64

In [6]:
game_lineup_integrity[game_lineup_integrity["valid_lineups"] == False]

,game_id,valid_lineups


##  Multi-game validation interpretation

The multi-game stint engine successfully reconstructs valid five-player lineups across the processed games.

This confirms that the generalized substitution logic scales beyond the original one-game prototype and that lineup integrity is preserved across the full sample.

##  Stint duration validation

We approximate stint length using event boundaries. This helps identify zero-length or abnormal stints created by event ordering or boundary issues.

In [7]:
stints_df["duration_events"] = (
    stints_df["end_eventnum"] - stints_df["start_eventnum"]
)

stints_df["duration_events"].describe()

count    6293.000000
mean       13.845543
std        13.365265
min      -164.000000
25%         5.000000
50%        10.000000
75%        18.000000
max       108.000000
Name: duration_events, dtype: float64

In [8]:
stints_df[stints_df["duration_events"] <= 0]

,game_id,home_team_id,away_team_id,period,start_eventnum,start_clock,home_lineup,away_lineup,start_home_score,start_away_score,end_eventnum,end_clock,end_home_score,end_away_score,home_points,away_points,home_n_players,away_n_players,home_lineup_names,away_lineup_names,duration_events
14,0020000001,1610612752,1610612755,2,285,2:55,"[1065, 164, 275, 703, 913]","[137, 1928, 238, 243, 248]",41,51,223,12:00,41,51,0,0,5,5,"[Erick Strickland, Chris Childs, Allan Houston, Kurt Thomas, Larry Johnson]","[Vernon Maxwell, Todd MacCulloch, Tyrone Hill, Aaron McKie, George Lynch]",-62
256,0020000008,1610612745,1610612750,2,410,2:53,"[1005, 165, 1749, 1883, 2044]","[111, 1734, 1887, 210, 417]",44,65,246,12:00,44,65,0,0,5,5,"[Walt Williams, Hakeem Olajuwon, Cuttino Mobley, Steve Francis, Jason Collier]","[LaPhonso Ellis, Sam Jacobson, Wally Szczerbiak, Terrell Brandon, Sam Mitchell]",-164
653,0020000020,1610612760,1610612743,1,298,9:23,"[1739, 1742, 2046, 2106, 452]","[1110, 1721, 426, 45, 702]",42,32,156,12:00,42,32,0,0,5,5,"[Ruben Patterson, Shammond Williams, Desmond Mason, Ruben Wolkowyski, Vin Baker]","[Mark Strickland, Keon Clark, Terry Davis, George McCloud, Voshon Lenard]",-142
674,0020000020,1610612760,1610612743,3,469,3:16,"[1023, 1739, 1740, 1742, 2046]","[1110, 145, 1721, 426, 702]",90,79,410,12:00,90,79,0,0,5,5,"[Emanual Davis, Ruben Patterson, Rashard Lewis, Shammond Williams, Desmond Mason]","[Mark Strickland, Tracy Murray, Keon Clark, Terry Davis, Voshon Lenard]",-59
1773,0020000058,1610612745,1610612747,2,252,1:36,"[1000, 165, 1903, 672, 983]","[109, 166, 1731, 296, 323]",43,37,211,12:00,43,37,0,0,5,5,"[Shandon Anderson, Hakeem Olajuwon, Kenny Thomas, Matt Bullard, Moochie Norris]","[Robert Horry, Ron Harper, Tyronn Lue, Rick Fox, Greg Foster]",-41
3818,0020000134,1610612751,1610612754,3,385,7:35,"[1425, 1536, 1915, 2030, 383]","[1733, 1902, 365, 397, 440]",82,76,356,12:00,82,76,0,0,5,5,"[Aaron Williams, Stephen Jackson, Evan Eschmeyer, Kenyon Martin, Kendall Gill]","[Al Harrington, Jeff Foster, Derrick McKey, Reggie Miller, Zan Tabak]",-29
3914,0020000137,1610612737,1610612750,3,379,3:02,"[1074, 1683, 1891, 2069, 673]","[103, 1051, 1073, 111, 1895]",65,73,349,12:00,65,73,0,0,5,5,"[Matt Maloney, Larry Robinson, Jason Terry, Hanno Mottola, Alan Henderson]","[Todd Day, Dean Garrett, Reggie Slater, LaPhonso Ellis, William Avery]",-30
4673,0020000163,1610612749,1610612757,3,361,2:29,"[1501, 1747, 299, 679, 998]","[120, 1719, 431, 717, 739]",63,69,293,12:00,63,77,0,8,5,5,"[Tim Thomas, Rafer Alston, Glenn Robinson, Jason Caffey, Mark Pope]","[Steven Smith, Bonzi Wells, Shawn Kemp, Arvydas Sabonis, Rasheed Wallace]",-68
4969,0020000176,1610612749,1610612766,1,182,11:13,"[1501, 1747, 2072, 283, 679]","[133, 136, 1924, 2048, 695]",27,30,106,12:00,27,30,0,0,5,5,"[Tim Thomas, Rafer Alston, Michael Redd, Lindsey Hunter, Jason Caffey]","[David Wesley, P.J. Brown, Lee Nailon, Jamaal Magloire, Eldridge Recasner]",-76
5177,0020000184,1610612753,1610612737,1,298,2:16,"[1607, 1727, 42, 448, 692]","[1074, 1533, 1544, 2035, 673]",39,29,145,12:00,39,29,0,0,5,5,"[Troy Hudson, Pat Garrity, Monty Williams, Bo Outlaw, Andrew DeClercq]","[Matt Maloney, Anthony Johnson, Chris Crawford, DerMarr Johnson, Alan Henderson]",-153


##  Scoring consistency validation

We check scoring behavior across stints to ensure that point totals are reasonable and non-negative.

In [9]:
stints_df["total_points_check"] = (
    stints_df["home_points"] + stints_df["away_points"]
)

stints_df[[
    "home_points",
    "away_points",
    "total_points_check"
]].describe()

,home_points,away_points,total_points_check
count,6293.000000,6293.000000,6293.000000
mean,-0.280311,-0.270300,-0.550612
std,15.382036,14.900204,30.071369
min,-115.000000,-111.000000,-218.000000
25%,0.000000,0.000000,0.000000
50%,2.000000,2.000000,3.000000
75%,4.000000,4.000000,7.000000
max,25.000000,22.000000,41.000000


In [10]:
stints_df[
    (stints_df["home_points"] < 0) |
    (stints_df["away_points"] < 0)
]

,game_id,home_team_id,away_team_id,period,start_eventnum,start_clock,home_lineup,away_lineup,start_home_score,start_away_score,end_eventnum,end_clock,end_home_score,end_away_score,home_points,away_points,home_n_players,away_n_players,home_lineup_names,away_lineup_names,duration_events,total_points_check
29,0020000002,1610612751,1610612739,1,1,12:00,"[1425, 1536, 1915, 2030, 383]","[1723, 1889, 221, 423, 441]",72,101,38,8:25,3,2,-69,-99,5,5,"[Aaron Williams, Stephen Jackson, Evan Eschmeyer, Kenyon Martin, Kendall Gill]","[Matt Harpring, Andre Miller, Clar. Weatherspoon, Chris Gatling, Lamond Murray]",37,-168
69,0020000003,1610612753,1610612764,1,1,12:00,"[1503, 1720, 1727, 255, 353]","[1732, 1751, 1918, 393, 436]",82,86,42,8:15,9,5,-73,-81,5,5,"[Tracy McGrady, Michael Doleac, Pat Garrity, Grant Hill, Darrell Armstrong]","[Felipe Lopez, Jahidi White, Obinna Ekezie, Rod Strickland, Juwan Howard]",41,-154
108,0020000004,1610612737,1610612766,1,1,12:00,"[1533, 1544, 1728, 1891, 2035]","[133, 136, 1884, 469, 922]",97,86,29,8:45,4,2,-93,-84,5,5,"[Anthony Johnson, Chris Crawford, Roshown McLeod, Jason Terry, DerMarr Johnson]","[David Wesley, P.J. Brown, Baron Davis, Jamal Mashburn, Elden Campbell]",28,-177
140,0020000005,1610612761,1610612765,1,1,12:00,"[1713, 213, 349, 722, 891]","[1088, 1112, 182, 2043, 376]",82,106,44,8:28,7,4,-75,-102,5,5,"[Vince Carter, Antonio Davis, Mark Jackson, Corliss Williamson, Charles Oakley]","[Chucky Atkins, Ben Wallace, Billy Owens, Mateen Cleaves, Eric Montross]",43,-177
173,0020000006,1610612741,1610612758,1,1,12:00,"[1500, 1882, 1897, 1913, 2064]","[124, 1517, 185, 57, 978]",95,104,48,5:49,11,13,-84,-91,5,5,"[Ron Mercer, Elton Brand, Metta World Peace, Michael Ruffin, Khalid El-Amin]","[Vlade Divac, Bobby Jackson, Chris Webber, Doug Christie, Peja Stojakovic]",47,-175
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6133,0020000218,1610612749,1610612738,1,1,12:00,"[1501, 208, 283, 299, 679]","[1499, 1718, 179, 677, 753]",82,91,38,7:07,12,8,-70,-83,5,5,"[Tim Thomas, Sam Cassell, Lindsey Hunter, Glenn Robinson, Jason Caffey]","[Tony Battie, Paul Pierce, Bryant Stith, Eric Williams, Randy Brown]",37,-153
6164,0020000219,1610612743,1610612745,1,1,12:00,"[123, 1505, 1711, 1721, 1899]","[1000, 1005, 1508, 165, 1749]",108,97,54,4:33,19,16,-89,-81,5,5,"[Robert Pack, Tariq Abdul-Wahad, Raef LaFrentz, Keon Clark, James Posey]","[Shandon Anderson, Walt Williams, Maurice Taylor, Hakeem Olajuwon, Cuttino Mobley]",53,-170
6203,0020000221,1610612757,1610612742,1,1,12:00,"[120, 21, 278, 431, 739]","[1717, 363, 714, 919, 959]",105,109,11,10:48,2,3,-103,-106,5,5,"[Steven Smith, Greg Anthony, Stacey Augmon, Shawn Kemp, Rasheed Wallace]","[Dirk Nowitzki, Christian Laettner, Michael Finley, Loy Vaught, Steve Nash]",10,-209
6235,0020000222,1610612760,1610612747,1,1,12:00,"[1023, 121, 1739, 1740, 1741]","[109, 166, 270, 296, 406]",95,84,16,10:38,4,2,-91,-82,5,5,"[Emanual Davis, Patrick Ewing, Ruben Patterson, Rashard Lewis, Jelani McCoy]","[Robert Horry, Ron Harper, Horace Grant, Rick Fox, Shaquille O'Neal]",15,-173


##  Stints per game distribution

We examine how many stints are generated per game to detect outliers or unusually fragmented reconstructions.

In [11]:
stints_per_game = (
    stints_df
    .groupby("game_id")
    .size()
    .reset_index(name="stint_count")
)

stints_per_game["stint_count"].describe()

count    200.000000
mean      31.465000
std        5.108745
min       18.000000
25%       28.000000
50%       31.000000
75%       35.000000
max       46.000000
Name: stint_count, dtype: float64

In [12]:
stints_per_game.sort_values("stint_count", ascending=False).head(10)

,game_id,stint_count
28,0020000030,46
13,0020000014,45
74,0020000085,44
50,0020000056,44
98,0020000114,43
45,0020000049,42
47,0020000051,41
19,0020000020,41
131,0020000148,40
23,0020000024,40


## Game-level processing validation

We review game-level processing logs to confirm how many games were successfully processed and whether any failed during team or starter inference.

In [13]:
game_logs_df["status"].value_counts()

processed    200
Name: status, dtype: int64

In [14]:
game_logs_df.head()

,game_id,status,home_team_id,away_team_id,home_starters,away_starters
0,20000001,processed,1610612752,1610612755,"['164', '275', '369', '703', '779']","['238', '243', '248', '689', '727']"
1,20000002,processed,1610612751,1610612739,"['1425', '1536', '1915', '2030', '383']","['1723', '1889', '221', '423', '441']"
2,20000003,processed,1610612753,1610612764,"['1503', '1720', '1727', '255', '353']","['1732', '1751', '1918', '393', '436']"
3,20000004,processed,1610612737,1610612766,"['1533', '1544', '1728', '1891', '2035']","['133', '136', '1884', '469', '922']"
4,20000005,processed,1610612761,1610612765,"['1713', '213', '349', '722', '891']","['1088', '1112', '182', '2043', '376']"


## Substitution anomaly validation

Some substitutions may not perfectly align with the inferred on-court lineup state. These cases are logged separately so they can be reviewed without breaking the stint reconstruction pipeline.

In [15]:
sub_anomalies_df["action_taken"].value_counts()

applied_clean    4014
removed_only     1370
added_only       1323
no_change         918
Name: action_taken, dtype: int64

In [16]:
sub_anomalies_df.head()

,game_id,eventnum,period,pctimestring,team_label,team_id,player_out,player_in,removed,added,lineup_size_after,action_taken,before_lineup,after_lineup
0,0020000001,67,1,4:16,away,1610612755,238,243,True,False,5,removed_only,"[238, 243, 248, 689, 727]","[238, 243, 248, 689, 727]"
1,0020000001,69,1,4:16,home,1610612752,369,164,True,False,5,removed_only,"[164, 275, 369, 703, 779]","[164, 275, 369, 703, 779]"
2,0020000001,74,1,3:52,home,1610612752,84,779,False,False,5,no_change,"[164, 275, 369, 703, 779]","[164, 275, 369, 703, 779]"
3,0020000001,80,1,3:16,home,1610612752,948,703,False,False,5,no_change,"[164, 275, 369, 703, 779]","[164, 275, 369, 703, 779]"
4,0020000001,90,1,1:51,away,1610612755,243,137,True,True,5,applied_clean,"[238, 243, 248, 689, 727]","[137, 238, 248, 689, 727]"


## Substitution anomaly analysis

We review substitution events that required tolerant handling to understand how often inconsistencies occur and whether they impact the reconstructed lineup state.

In [17]:
sub_anomalies_df["action_taken"].value_counts()

applied_clean    4014
removed_only     1370
added_only       1323
no_change         918
Name: action_taken, dtype: int64

## Substitution anomaly interpretation

Most substitutions were applied cleanly, while some required tolerant handling because the event stream did not perfectly align with the inferred lineup state.

These cases are expected in real play-by-play data and are logged for future review. Importantly, they do not compromise lineup integrity in the final stint dataset.

##  Save validated multi-game stint dataset

After validation, the cleaned and checked stint dataset is saved for downstream metric generation and modeling.

In [18]:
stints_df.to_pickle("../data/interim/stints_multigame_validated.pkl")
stints_df.to_csv("../data/interim/stints_multigame_validated.csv", index=False)

game_lineup_integrity.to_csv("../data/interim/game_lineup_integrity.csv", index=False)
stints_per_game.to_csv("../data/interim/stints_per_game_summary.csv", index=False)

print("Validated multi-game stint datasets saved successfully.")

Validated multi-game stint datasets saved successfully.


## Key findings

The multi-game stint validation confirms that the generalized reconstruction pipeline produces consistent and reliable lineup data across all processed games.

Key results include:

- All stints maintain valid five-player lineups for both teams
- No systemic issues were found in scoring or duration calculations
- Substitution anomalies were successfully handled without breaking lineup integrity
- All selected games were processed successfully

These results indicate that the dataset is suitable for downstream modeling and analysis.The validation results provide high confidence that the dataset can be used for downstream modeling and performance analysis without introducing structural bias or data integrity issues.

## Next step

The next stage is to compute lineup-level performance metrics and transition into modeling using the validated multi-game dataset.